# Build `prefixdistances.json`

Creates a cumulative distance lookup (meters) for every trip/stop pair,
matching the format of `prefixtimes.json`:

```json
{ trip_id: { stop_id: cumulative_meters, ... }, ... }
```

**Pipeline:**
1. Load `shapes.txt` → compute cumulative Haversine distance along each shape
2. Load `stops.txt` → get stop coordinates
3. Load `trips.txt` → map trip_id → shape_id
4. Load `stop_times.txt` → get ordered stops per trip
5. For each trip, snap each stop to the nearest shape point → record cumulative distance
6. Save as `prefixdistances.json`

In [1]:
import pandas as pd
import json
import math
import numpy as np
from pathlib import Path

GTFS_DIR = Path("gtfsAlex")
OUTPUT_PATH = Path("prefixdistances.json")

## 1. Haversine helper

In [2]:
def haversine(lat1, lon1, lat2, lon2):
    """Distance in meters between two (lat, lon) points."""
    R = 6_371_000  # Earth radius in meters
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2) ** 2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

## 2. Load GTFS files

In [3]:
shapes_df = pd.read_csv(GTFS_DIR / "shapes.txt")
stops_df = pd.read_csv(GTFS_DIR / "stops.txt")
trips_df = pd.read_csv(GTFS_DIR / "trips.txt")
stop_times_df = pd.read_csv(GTFS_DIR / "stop_times.txt")

print(f"Shapes: {len(shapes_df)} points across {shapes_df['shape_id'].nunique()} shapes")
print(f"Stops: {len(stops_df)}")
print(f"Trips: {len(trips_df)}")
print(f"Stop-times: {len(stop_times_df)}")

Shapes: 29359 points across 192 shapes
Stops: 441
Trips: 192
Stop-times: 2548


## 3. Compute cumulative distance along each shape

In [4]:
# Sort shape points and compute cumulative distance
shapes_df = shapes_df.sort_values(["shape_id", "shape_pt_sequence"]).reset_index(drop=True)

# For each shape: list of (lat, lon, cumulative_dist)
shape_cum_dist = {}  # shape_id -> [(lat, lon, cum_dist), ...]

for shape_id, group in shapes_df.groupby("shape_id"):
    pts = group[["shape_pt_lat", "shape_pt_lon"]].values
    cum = [0.0]
    for i in range(1, len(pts)):
        d = haversine(pts[i-1][0], pts[i-1][1], pts[i][0], pts[i][1])
        cum.append(cum[-1] + d)
    shape_cum_dist[shape_id] = list(zip(pts[:, 0], pts[:, 1], cum))

# Quick check
example_shape = next(iter(shape_cum_dist))
pts = shape_cum_dist[example_shape]
print(f"Example shape '{example_shape}': {len(pts)} points, total length = {pts[-1][2]:.0f} m")

Example shape '-Q2gVX9GgVSEtVr-yv6Nf_Shape': 35 points, total length = 2745 m


## 4. Build lookup tables

In [5]:
# trip_id -> shape_id
trip_to_shape = dict(zip(trips_df["trip_id"].astype(str), trips_df["shape_id"].astype(str)))

# stop_id -> (lat, lon)
stop_coords = dict(zip(
    stops_df["stop_id"].astype(str),
    zip(stops_df["stop_lat"], stops_df["stop_lon"])
))

print(f"trip_to_shape: {len(trip_to_shape)} trips")
print(f"stop_coords: {len(stop_coords)} stops")

trip_to_shape: 192 trips
stop_coords: 441 stops


## 5. For each trip, snap stops to shape → get cumulative distance

We precompute numpy arrays per shape for fast vectorised nearest-point lookup.

In [6]:
# Pre-build numpy arrays for each shape (for fast snapping)
shape_arrays = {}  # shape_id -> (lats_arr, lons_arr, cum_arr)
for sid, pts in shape_cum_dist.items():
    arr = np.array(pts)  # shape (N, 3)
    shape_arrays[sid] = (arr[:, 0], arr[:, 1], arr[:, 2])

def snap_stop_to_shape(stop_lat, stop_lon, shape_lats, shape_lons, shape_cums):
    """Find the cumulative distance of the nearest shape point to a stop."""
    dlat = shape_lats - stop_lat
    dlon = shape_lons - stop_lon
    # Squared Euclidean in degree space (fine for nearest-neighbour at city scale)
    sq_dist = dlat * dlat + dlon * dlon
    idx = np.argmin(sq_dist)
    return shape_cums[idx]

print("Shape arrays ready.")

Shape arrays ready.


In [7]:
# Build prefixdistances
prefix_distances = {}
skipped = 0

stop_times_df["stop_id"] = stop_times_df["stop_id"].astype(str)
stop_times_df["trip_id"] = stop_times_df["trip_id"].astype(str)

for trip_id, group in stop_times_df.groupby("trip_id"):
    shape_id = trip_to_shape.get(trip_id)
    if shape_id is None or shape_id not in shape_arrays:
        skipped += 1
        continue

    s_lats, s_lons, s_cums = shape_arrays[shape_id]
    ordered_stops = group.sort_values("stop_sequence")

    trip_entry = {}
    first_cum = None

    for _, row in ordered_stops.iterrows():
        sid = row["stop_id"]
        coords = stop_coords.get(sid)
        if coords is None:
            continue
        cum = snap_stop_to_shape(coords[0], coords[1], s_lats, s_lons, s_cums)
        if first_cum is None:
            first_cum = cum
        # Store relative to first stop (so first stop = 0)
        trip_entry[sid] = round(abs(cum - first_cum))

    if trip_entry:
        prefix_distances[trip_id] = trip_entry

print(f"Built prefix distances for {len(prefix_distances)} trips (skipped {skipped})")

Built prefix distances for 192 trips (skipped 0)


## 6. Quick sanity check

Compare with `prefixtimes.json` to make sure we cover the same trips.

In [8]:
with open("prefixtimes.json") as f:
    prefix_times = json.load(f)

time_trips = set(prefix_times.keys())
dist_trips = set(prefix_distances.keys())

print(f"prefixtimes trips:     {len(time_trips)}")
print(f"prefixdistances trips: {len(dist_trips)}")
print(f"In both:               {len(time_trips & dist_trips)}")
print(f"In times but not dist: {len(time_trips - dist_trips)}")
print(f"In dist but not times: {len(dist_trips - time_trips)}")

# Show a sample entry
sample_trip = next(iter(prefix_distances))
print(f"\nSample trip: {sample_trip}")
print(f"  distances: {prefix_distances[sample_trip]}")
if sample_trip in prefix_times:
    print(f"  times:     {prefix_times[sample_trip]}")

prefixtimes trips:     192
prefixdistances trips: 192
In both:               192
In times but not dist: 0
In dist but not times: 0

Sample trip: -Q2gVX9GgVSEtVr-yv6Nf-07:00:00
  distances: {'309': 0, '310': 555, '443': 1428}
  times:     {'309': 0, '310': 68, '443': 346}


## 7. Save

In [9]:
with open(OUTPUT_PATH, "w") as f:
    json.dump(prefix_distances, f, indent=2)

size_mb = OUTPUT_PATH.stat().st_size / 1_048_576
print(f"Saved to {OUTPUT_PATH} ({size_mb:.2f} MB)")

Saved to prefixdistances.json (0.05 MB)
